In [1]:
#Churn prediction in telecom.
import numpy as np
import matplotlib.pyplot as plt

In [2]:
!gdown 1uUt7uL-VuF_5cpodYRiriEwhsldeEp3m

Downloading...
From: https://drive.google.com/uc?id=1uUt7uL-VuF_5cpodYRiriEwhsldeEp3m
To: /content/churn_logistic.csv
100% 494k/494k [00:00<00:00, 43.0MB/s]


In [3]:
import pandas as pd
churn = pd.read_csv("churn_logistic.csv")
churn.head()

,Account Length,VMail Message,Day Mins,Eve Mins,Night Mins,Intl Mins,CustServ Calls,Intl Plan,VMail Plan,Day Calls,...,Eve Calls,Eve Charge,Night Calls,Night Charge,Intl Calls,Intl Charge,State,Area Code,Phone,Churn
0,128,25,265.1,197.4,244.7,10.0,1,0,1,110,...,99,16.78,91,11.01,3,2.70,KS,415,382-4657,0
1,107,26,161.6,195.5,254.4,13.7,1,0,1,123,...,103,16.62,103,11.45,3,3.70,OH,415,371-7191,0
2,137,0,243.4,121.2,162.6,12.2,0,0,0,114,...,110,10.30,104,7.32,5,3.29,NJ,415,358-1921,0
3,84,0,299.4,61.9,196.9,6.6,2,1,0,71,...,88,5.26,89,8.86,7,1.78,OH,408,375-9999,0
4,75,0,166.7,148.3,186.9,10.1,3,1,0,113,...,122,12.61,121,8.41,3,2.73,OK,415,330-6626,0


In [4]:
# Check the shape and columns of the dataset
print("Shape:", churn.shape)
print("\nColumns:", churn.columns.tolist())
print("\nInfo:")
churn.info()

Shape: (5700, 21)

Columns: ['Account Length', 'VMail Message', 'Day Mins', 'Eve Mins', 'Night Mins', 'Intl Mins', 'CustServ Calls', 'Intl Plan', 'VMail Plan', 'Day Calls', 'Day Charge', 'Eve Calls', 'Eve Charge', 'Night Calls', 'Night Charge', 'Intl Calls', 'Intl Charge', 'State', 'Area Code', 'Phone', 'Churn']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5700 entries, 0 to 5699
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Account Length  5700 non-null   int64  
 1   VMail Message   5700 non-null   int64  
 2   Day Mins        5700 non-null   float64
 3   Eve Mins        5700 non-null   float64
 4   Night Mins      5700 non-null   float64
 5   Intl Mins       5700 non-null   float64
 6   CustServ Calls  5700 non-null   int64  
 7   Intl Plan       5700 non-null   int64  
 8   VMail Plan      5700 non-null   int64  
 9   Day Calls       5700 non-null   int64  
 10  Day Charge      5700 non-nu

In [5]:
# Prepare features and target
# Drop non-numeric columns (State, Phone) and target column
cols = ['Day Mins', 'Eve Mins', 'CustServ Calls', 'Intl Plan','VMail Message']
y = churn["Churn"]
X = churn[cols]


print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

Features shape: (5700, 5)
Target shape: (5700,)

Target distribution:
Churn
0    2850
1    2850
Name: count, dtype: int64


In [6]:
# Create Train, Validation, and Test splits
# First split: 80% train+val, 20% test
# Second split: From 80%, take 75% train (60% of total), 25% val (20% of total)
from sklearn.model_selection import train_test_split

# First split: separate test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Second split: separate validation set (25% of remaining = 20% of total)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print("Train set size:", X_train.shape[0], f"({X_train.shape[0]/len(X)*100:.1f}%)")
print("Validation set size:", X_val.shape[0], f"({X_val.shape[0]/len(X)*100:.1f}%)")
print("Test set size:", X_test.shape[0], f"({X_test.shape[0]/len(X)*100:.1f}%)")

Train set size: 3420 (60.0%)
Validation set size: 1140 (20.0%)
Test set size: 1140 (20.0%)


In [7]:
# Standardization: Fit on train, transform train, validation, and test
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit on training data only
scaler.fit(X_train)

# Transform all three sets
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)



In [8]:
# Train Vanilla Logistic Regression
from sklearn.linear_model import LogisticRegression

# Initialize and train the model
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)



LogisticRegression(random_state=42)

In [9]:
# Evaluate on VALIDATION set first
from sklearn.metrics import accuracy_score

# Predictions on validation set
y_val_pred = log_reg.predict(X_val_scaled)

print("=" * 50)
print("VALIDATION SET METRICS")
print("=" * 50)
print(f"\nAccuracy: {accuracy_score(y_val, y_val_pred):.4f}")

VALIDATION SET METRICS

Accuracy: 0.7535


In [10]:
# After confirming on validation, evaluate on TEST set
# Predictions on test set
y_test_pred = log_reg.predict(X_test_scaled)

print("=" * 50)
print("TEST SET METRICS (Final Evaluation)")
print("=" * 50)
print(f"\nAccuracy: {accuracy_score(y_test, y_test_pred):.4f}")

TEST SET METRICS (Final Evaluation)

Accuracy: 0.7307


In [ ]:
# Summary: Compare Validation vs Test metrics
print("=" * 60)
print("SUMMARY: Validation vs Test Performance Comparison")
print("=" * 60)

val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"\nValidation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy:       {test_acc:.4f}")
print(f"Difference:          {abs(val_acc-test_acc):.4f}")

if abs(val_acc - test_acc) < 0.02:
    print("\n✓ Model generalizes well - validation and test accuracy are consistent!")

## Understanding C vs Lambda (λ) in Regularization

### In Regularized Linear Regression (Ridge/Lasso):
The cost function is:
$$J(\theta) = \text{Loss} + \lambda \cdot \text{Regularization Term}$$

Where **λ (lambda)** is the regularization strength:
- **Higher λ** → More regularization → Simpler model (prevents overfitting)
- **Lower λ** → Less regularization → Model can fit training data more closely

### In Logistic Regression (sklearn):
The cost function is:
$$J(\theta) = C \cdot \text{Loss} + \text{Regularization Term}$$

Where **C = 1/λ** (inverse relationship):
- **Higher C** → Less regularization → Model focuses more on fitting training data
- **Lower C** → More regularization → Simpler model (prevents overfitting)

### Key Relationship:
$$C = \frac{1}{\lambda}$$

| C value | λ value | Effect |
|---------|---------|--------|
| Large (e.g., 100) | Small (0.01) | Less regularization, risk of overfitting |
| Small (e.g., 0.01) | Large (100) | More regularization, risk of underfitting |
| Default (1.0) | 1.0 | Balanced regularization |

In [ ]:
# Finding the optimal C using Validation data
# We'll try a range of C values and evaluate on validation set

C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

# Store results
results = []

print("Finding optimal C using VALIDATION set")
print("=" * 55)
print(f"{'C':<10} {'Lambda (1/C)':<15} {'Val Accuracy':<15}")
print("-" * 55)

for C in C_values:
    # Train model with this C value
    model = LogisticRegression(C=C, random_state=42, max_iter=1000)
    model.fit(X_train_scaled, y_train)

    # Evaluate on validation set
    y_val_pred = model.predict(X_val_scaled)
    val_acc = accuracy_score(y_val, y_val_pred)

    results.append({
        'C': C,
        'lambda': 1/C,
        'val_accuracy': val_acc,
        'model': model
    })

    print(f"{C:<10} {1/C:<15.4f} {val_acc:<15.4f}")

# Find best C based on validation accuracy
best_result = max(results, key=lambda x: x['val_accuracy'])
best_C = best_result['C']

print("-" * 55)
print(f"\n✓ Best C = {best_C} (λ = {1/best_C:.4f}) with Validation Accuracy = {best_result['val_accuracy']:.4f}")

In [ ]:
# Confirm the best C on TEST set
# Use the best model we already found
best_model = best_result['model']

# Predict on test set
y_test_pred_final = best_model.predict(X_test_scaled)
test_acc_final = accuracy_score(y_test, y_test_pred_final)

# Get validation accuracy we already computed
val_acc_final = best_result['val_accuracy']

print("=" * 60)
print(f"CONFIRMING BEST C = {best_C} ON TEST SET")
print("=" * 60)

print(f"\nValidation Accuracy (already computed): {val_acc_final:.4f}")
print(f"Test Accuracy:                          {test_acc_final:.4f}")


In [ ]:
# Final Summary with optimal C
print("=" * 60)
print(f"FINAL SUMMARY (Best C = {best_C})")
print("=" * 60)

print(f"\n📊 Final Results:")
print(f"   - Validation Accuracy: {val_acc_final:.4f} ({val_acc_final*100:.2f}%)")
print(f"   - Test Accuracy:       {test_acc_final:.4f} ({test_acc_final*100:.2f}%)")

print(f"\n📝 Model Configuration:")
print(f"   - C = {best_C} (regularization parameter)")
print(f"   - λ = {1/best_C:.4f} (equivalent lambda)")

print(f"\n✓ The model correctly classifies {test_acc_final*100:.2f}% of the test samples!")

## K-Fold vs Stratified K-Fold Cross-Validation

To clearly demonstrate the difference, we'll create a **synthetic imbalanced dataset** where one class is rare (10%).

**K-Fold**: Splits data randomly - class proportions can vary significantly between folds

**Stratified K-Fold**: Ensures each fold maintains the same class proportions as original data

In [ ]:
# Create a synthetic IMBALANCED dataset for demonstration
from sklearn.datasets import make_classification

# Create imbalanced data: 90% class 0, 10% class 1
X_imb, y_imb = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=3,
    n_redundant=1,
    n_classes=2,
    weights=[0.9, 0.1],  # 90% class 0, 10% class 1 (imbalanced!)
    random_state=42
)

# Convert to DataFrame for easier handling
X_imb = pd.DataFrame(X_imb, columns=[f'Feature_{i}' for i in range(5)])
y_imb = pd.Series(y_imb)

print("Synthetic Imbalanced Dataset Created")
print("=" * 50)
print(f"Total samples: {len(y_imb)}")
print(f"Class 0: {(y_imb == 0).sum()} samples ({(y_imb == 0).mean()*100:.1f}%)")
print(f"Class 1: {(y_imb == 1).sum()} samples ({(y_imb == 1).mean()*100:.1f}%)")
print("\n⚠ This is an IMBALANCED dataset - Class 1 is rare!")
display(X_imb.head())

In [ ]:
# Regular K-Fold on imbalanced data - see the problem!
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

print("Regular K-Fold on IMBALANCED data")
print("=" * 70)
print("Original Data: Class 0 (Majority) = 900, Class 1 (Minority) = 100")
print("Original Ratio (Class0 : Class1) = 9 : 1")
print("-" * 70)
print("\nValidation set composition in each fold:")
print(f"{'Class 0':<10} {'Class 1':<10} {'Ratio (C0:C1)':<15}")
print("-" * 35)

for train_idx, val_idx in kfold.split(X_imb):
    val_class0 = (y_imb.iloc[val_idx] == 0).sum()
    val_class1 = (y_imb.iloc[val_idx] == 1).sum()
    ratio = val_class0 / val_class1 if val_class1 > 0 else 0
    print(f"{val_class0:<10} {val_class1:<10} {ratio:.1f} : 1")

print("\n⚠ Problem: Ratios vary across folds! (not always 9:1)")

## Stratified K-Fold - The Solution

**Stratified K-Fold** ensures each fold has the **same percentage of each class** as the original dataset.

Key difference in code: `skfold.split(X, y)` - we pass **y** so it knows the class labels!

In [ ]:
# Stratified K-Fold on imbalanced data - class balance preserved!
from sklearn.model_selection import StratifiedKFold

skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold on IMBALANCED data")
print("=" * 70)
print("Original Data: Class 0 (Majority) = 900, Class 1 (Minority) = 100")
print("Original Ratio (Class0 : Class1) = 9 : 1")
print("-" * 70)
print("\nValidation set composition in each fold:")
print(f"{'Class 0':<10} {'Class 1':<10} {'Ratio (C0:C1)':<15}")
print("-" * 35)

for train_idx, val_idx in skfold.split(X_imb, y_imb):  # Note: y_imb passed!
    val_class0 = (y_imb.iloc[val_idx] == 0).sum()
    val_class1 = (y_imb.iloc[val_idx] == 1).sum()
    ratio = val_class0 / val_class1 if val_class1 > 0 else 0
    print(f"{val_class0:<10} {val_class1:<10} {ratio:.1f} : 1")

print("\n✓ Solution: Ratio is consistently 9:1 in every fold!")

In [ ]:
# Compare model performance: K-Fold vs Stratified K-Fold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# K-Fold accuracies
kfold_accs = []
for train_idx, val_idx in kfold.split(X_imb):
    X_tr, X_vl = X_imb.iloc[train_idx], X_imb.iloc[val_idx]
    y_tr, y_vl = y_imb.iloc[train_idx], y_imb.iloc[val_idx]

    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_vl_sc = scaler.transform(X_vl)

    model = LogisticRegression(random_state=42)
    model.fit(X_tr_sc, y_tr)
    kfold_accs.append(accuracy_score(y_vl, model.predict(X_vl_sc)))

# Stratified K-Fold accuracies
skfold_accs = []
for train_idx, val_idx in skfold.split(X_imb, y_imb):
    X_tr, X_vl = X_imb.iloc[train_idx], X_imb.iloc[val_idx]
    y_tr, y_vl = y_imb.iloc[train_idx], y_imb.iloc[val_idx]

    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_vl_sc = scaler.transform(X_vl)

    model = LogisticRegression(random_state=42)
    model.fit(X_tr_sc, y_tr)
    skfold_accs.append(accuracy_score(y_vl, model.predict(X_vl_sc)))

print("Model Performance Comparison")
print("=" * 50)
print(f"\n{'Method':<20} {'Mean Acc':<12} {'Std Dev':<12}")
print("-" * 45)
print(f"{'K-Fold':<20} {np.mean(kfold_accs):.4f}       {np.std(kfold_accs):.4f}")
print(f"{'Stratified K-Fold':<20} {np.mean(skfold_accs):.4f}       {np.std(skfold_accs):.4f}")

In [ ]:
# Key Takeaways
print("=" * 55)
print("KEY TAKEAWAYS")
print("=" * 55)

print("""
1. K-Fold: Splits randomly, class proportions can vary
   - Problem: Some folds may have very few minority samples
   - Results can be inconsistent/unreliable

2. Stratified K-Fold: Preserves class proportions in each fold
   - Solution: Each fold has same class % as original data
   - Results are more consistent and reliable

3. When to use Stratified K-Fold:
   - Classification problems (especially imbalanced data)
   - When you want consistent evaluation across folds

4. Code difference:
   - KFold:          kfold.split(X)
   - StratifiedKFold: skfold.split(X, y)  ← pass y!
""")